# Task 1 — Profile the Data

Establish exactly how messy `raw_pos_export.csv` is, with real counts, before assuming anything about its structure. Every finding here directly shapes the assumption tests in `02_validate_assumptions.ipynb` and the reconstruction rule in `03_reconstruct_invoices.ipynb`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw_pos_export.csv")

# check: confirms the file loaded as expected before profiling anything
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   row_id              131 non-null    int64
 1   store_id            131 non-null    str  
 2   register_id         131 non-null    str  
 3   cashier_name        131 non-null    str  
 4   transaction_ts      130 non-null    str  
 5   invoice_number      38 non-null     str  
 6   customer_ref        117 non-null    str  
 7   customer_name_raw   117 non-null    str  
 8   customer_email_raw  116 non-null    str  
 9   product_sku         129 non-null    str  
 10  product_name_raw    129 non-null    str  
 11  category_raw        131 non-null    str  
 12  qty                 131 non-null    int64
 13  unit_price_raw      128 non-null    str  
 14  discount_code       66 non-null     str  
 15  payment_method      131 non-null    str  
 16  line_note           7 non-null      str  
dtypes: int64

## Null rates

Raw counts are hard to read at a glance -- convert to percentages.

In [2]:
# null rate per column, as a percentage, easier to read than raw counts
null_rates = (df.isnull().mean() * 100).round(1)

# check: invoice_number and line_note should stand out as mostly empty
print(null_rates)

row_id                 0.0
store_id               0.0
register_id            0.0
cashier_name           0.0
transaction_ts         0.8
invoice_number        71.0
customer_ref          10.7
customer_name_raw     10.7
customer_email_raw    11.5
product_sku            1.5
product_name_raw       1.5
category_raw           0.0
qty                    0.0
unit_price_raw         2.3
discount_code         49.6
payment_method         0.0
line_note             94.7
dtype: float64


**Finding:** `invoice_number` is populated on only ~29% of rows -- not usable as a reliable grouping key on its own. `line_note` is populated on only ~5% of rows, which is expected (it's a free-text field only filled in for exceptions like returns or gift wrap).

## Duplicate rows

In [3]:
# naive check across all columns, including row_id
naive_dupes = df.duplicated().sum()

# check: this comes back 0 -- but that's misleading, see next cell
print("duplicates including row_id:", naive_dupes)

duplicates including row_id: 0


In [4]:
# row_id is a sequential export number, not a business key -- it makes
# every row look "unique" even when the real transaction data is identical.
# drop it before checking for true duplicates
real_dupes = df.drop(columns="row_id").duplicated().sum()

# check: this is the real duplicate count, likely a POS double-submit glitch
print("duplicates excluding row_id:", real_dupes)

duplicates excluding row_id: 2


**Finding:** 2 exact duplicate rows once `row_id` is excluded -- the naive check hid this completely.

## Distinct value counts

In [5]:
# distinct value counts for the columns most likely to carry inconsistency
counts = df[["store_id", "category_raw", "payment_method"]].nunique()

# check: category_raw showing more values than expected is a red flag
print(counts)

store_id          4
category_raw      8
payment_method    5
dtype: int64


In [6]:
# does case-folding collapse category_raw down to fewer real categories?
case_folded_count = df["category_raw"].str.lower().nunique()

# check: if this is lower than the raw count above, it's a casing problem,
# not genuinely different categories
print("distinct categories after lowercasing:", case_folded_count)

distinct categories after lowercasing: 4


**Finding:** `category_raw` shows 8 distinct values but collapses to 4 once case-folded (e.g. `Home` vs `HOME`) -- inconsistent casing, not real categories.

## Store reference reconciliation

Compare store IDs in the export against the reference table -- a preview of the migration-style reconciliation this project is meant to practice.

In [7]:
# load the store reference table so we can compare it against the export
stores = pd.read_csv("../data/store_lookup.csv")

# set difference: store_ids present in the export but missing from the
# lookup table -- these are unmapped stores we can't get a name for
print("in export but not lookup:", set(df["store_id"]) - set(stores["store_id"]))

# the reverse: store_ids in the lookup table that never actually show up
# in the export -- these are reference entries with no matching activity
print("in lookup but not export:", set(stores["store_id"]) - set(df["store_id"]))

in export but not lookup: {'ST99'}
in lookup but not export: {'ST04'}


**Finding:** `ST99` appears in the export but not in the lookup table; `ST04` exists in the lookup table but is never used in the export.

## Summary

| Check | Result |
|---|---|
| Rows / columns | 131 rows, 17 columns |
| `invoice_number` completeness | ~29% populated -- not a reliable grouping key |
| `line_note` completeness | ~5% populated -- expected, exception-only field |
| Exact duplicates | 2, hidden by `row_id` unless explicitly excluded |
| `category_raw` | 8 raw values, 4 real categories once case-folded |
| Store reconciliation | `ST99` unmapped in export; `ST04` unused in lookup |

These findings carry forward directly: the `invoice_number` gap motivates Task 2's assumption testing and Task 3's reconstruction rule; the store mismatch feeds Task 5's reconciliation step.